# Notebook 6 – Loss Functions


## Setup

In [2]:
import numpy as np
import torch
import torch.nn as nn


## Why Loss Functions Are Required

**Explain:** A **loss function** measures **how wrong** a model's prediction is compared to the true answer, as a single number. Without a loss, the network has no way to know whether it's improving, and no signal to guide **backpropagation** (covered in the next notebook) on how to adjust its weights.

**Analogy:** A loss function is like a golf score — a single number that tells you how far you are from the hole (the correct answer). Without that score, you'd have no idea whether your last swing helped or hurt.

**Implement:** compare two predictions for the same true value, using a loss.

In [3]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)
y_true = np.array([10.0])
print("Bad prediction (7.0)  loss:", mse(y_true, np.array([7.0])))
print("Good prediction (9.5) loss:", mse(y_true, np.array([9.5])))

Bad prediction (7.0)  loss: 9.0
Good prediction (9.5) loss: 0.25


**Interpret:** The closer prediction gets a much smaller loss. This single number is exactly what training tries to **minimize** — every weight update is aimed at making this number smaller.


# Part 1: Regression Losses

## 1. MAE (Mean Absolute Error)

**Definition:** The average of the absolute differences between predicted and true values.

**Formula:** `MAE = (1/n) * sum(|y_true - y_pred|)`

**Mathematical intuition:** Treats every error the same way regardless of size — an error of 10 counts exactly 10x as much as an error of 1. It answers "on average, how far off am I, in the original units?"

**Simple numerical example:**

In [5]:
y_true = np.array([3.0, 5.0, 2.0])
y_pred = np.array([2.5, 6.0, 2.0])
print("Errors:", y_true - y_pred)   

Errors: [ 0.5 -1.   0. ]


**Python implementation:**

In [6]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))
print("MAE:", mae(y_true, y_pred))

MAE: 0.5


**PyTorch implementation:**

In [7]:
loss_fn = nn.L1Loss()     
print("MAE (PyTorch):", loss_fn(torch.tensor(y_pred), torch.tensor(y_true)).item())

MAE (PyTorch): 0.5


**Interpretation:** On average, predictions are off by 0.5 units — in the *same units* as the original target, which makes it easy to explain to non-technical stakeholders.

**When to use:** when all errors should be weighted equally, or when the data has **outliers** you don't want to dominate the loss.

**When not to use:** when large errors should be penalized much more heavily than small ones (use MSE instead).

**Advantages:** easy to interpret, robust to outliers.

**Limitations:** its gradient has a constant size (doesn't shrink near the minimum), which can make fine-tuning near the optimum less smooth than MSE.

## 2. MSE (Mean Squared Error)

**Definition:** The average of the **squared** differences between predicted and true values.

**Formula:** `MSE = (1/n) * sum((y_true - y_pred)^2)`

**Mathematical intuition:** Squaring makes larger errors count **much** more than small ones (an error of 10 contributes 100x more than an error of 1) — this pushes the model hard to avoid big mistakes.

**Simple numerical example:**

In [8]:
print("Squared errors:", (y_true - y_pred) ** 2)  

Squared errors: [0.25 1.   0.  ]


**Python implementation:**

In [9]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)
print("MSE:", mse(y_true, y_pred))

MSE: 0.4166666666666667


**PyTorch implementation:**

In [10]:
loss_fn = nn.MSELoss()
print("MSE (PyTorch):", loss_fn(torch.tensor(y_pred), torch.tensor(y_true)).item())

MSE (PyTorch): 0.4166666666666667


**Interpretation:** MSE (0.417) is larger than MAE (0.5) would suggest at a glance because it's in **squared units** — it's less directly interpretable than MAE, but it strongly discourages big errors.

**When to use:** when large errors are especially undesirable (e.g. predicting a critical quantity where big misses are costly), and there are no extreme outliers distorting the data.

**When not to use:** when the data has outliers — a single huge error can dominate and skew the whole loss.

**Advantages:** smooth gradient that shrinks as predictions improve, mathematically convenient (used in classic linear regression).

**Limitations:** very sensitive to outliers; not in the same units as the original target (squared units).

## 3. RMSE (Root Mean Squared Error)

**Definition:** The square root of MSE — brings the error back into the **original units**.

**Formula:** `RMSE = sqrt(MSE)`

**Mathematical intuition:** Same emphasis on large errors as MSE, but the final number is directly comparable to the target's own scale again.

**Simple numerical example / Python implementation:**

In [11]:
def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))
print("RMSE:", rmse(y_true, y_pred))

RMSE: 0.6454972243679028


**PyTorch implementation:** (PyTorch has no built-in RMSE — just take the square root of `MSELoss`)

In [12]:
mse_loss = nn.MSELoss()(torch.tensor(y_pred), torch.tensor(y_true))
print("RMSE (PyTorch):", torch.sqrt(mse_loss).item())

RMSE (PyTorch): 0.6454972243679028


**Interpretation:** RMSE (0.645) is in the same units as the target, like MAE, but — because of the underlying squaring — still penalizes large errors more heavily.

**When to use:** when you want MSE's sensitivity to large errors, but need a number interpretable in the original units (very common choice for regression reporting).

**When not to use:** same caution as MSE — avoid if outliers would unfairly dominate the score.

**Advantages:** same-unit interpretability as MAE, combined with MSE's emphasis on large errors.

**Limitations:** still sensitive to outliers; slightly more expensive to compute (a square root) than MSE alone.


# Part 2: Binary Classification Loss

## 4. Binary Cross Entropy (BCE)

**Definition:** Measures how well predicted **probabilities** (between 0 and 1) match true binary labels (0 or 1) — the standard loss for binary classification.

**Formula:** `BCE = -(1/n) * sum( y_true*log(y_pred) + (1-y_true)*log(1-y_pred) )`

**Mathematical intuition:** If the true label is 1, the loss is `-log(y_pred)` — this is **small** when `y_pred` is close to 1, and shoots toward **infinity** as `y_pred` approaches 0 (a confident wrong answer is punished very heavily). The mirror logic applies when the true label is 0.

**Simple numerical example:**

In [14]:
y_true_bin = np.array([1, 0, 1])
y_pred_prob = np.array([0.9, 0.1, 0.2])   

**Python implementation:**

In [15]:
def bce(y_true, y_pred, eps=1e-9):
    y_pred = np.clip(y_pred, eps, 1 - eps)   
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
print("BCE:", bce(y_true_bin, y_pred_prob))

BCE: 0.6067196479165843


**PyTorch implementation:**

In [16]:
loss_fn = nn.BCELoss()   
print("BCE (PyTorch):", loss_fn(torch.tensor(y_pred_prob, dtype=torch.float32), torch.tensor(y_true_bin, dtype=torch.float32)).item())

BCE (PyTorch): 0.6067196726799011


**Interpretation:** The loss is dominated by the 3rd example, where the model confidently predicted 0.2 for a true label of 1 — a small change there would reduce the total loss far more than tweaking the already-good first two predictions.

**When to use:** any binary (2-class) classification problem, whenever predictions are probabilities.

**When not to use:** multiclass problems (use Cross Entropy Loss below instead), or regression tasks.

**Advantages:** directly tied to probability, heavily penalizes confident wrong answers (which is usually exactly what we want).

**Limitations:** requires predictions to actually be valid probabilities (0 to 1) — feeding it raw, unbounded scores breaks the formula (see the next section, BCE with logits).

## 5. BCE With Logits (Concept)

**Explain:** A **logit** is the network's **raw output before** the sigmoid activation (any real number, not yet a 0–1 probability). `BCEWithLogitsLoss` combines the **sigmoid activation and BCE into one step**, which is more numerically stable than applying sigmoid yourself and then calling plain `BCELoss` — it avoids computing `log(0)` errors that can occur when sigmoid outputs get extremely close to 0 or 1.

**Analogy:** Rather than measuring a runner's exact finishing speed and then separately calculating their time (two error-prone steps), a combined finish-line sensor calculates the time directly and more precisely in one integrated step.

**Implement:**

In [17]:
logits = np.array([2.0, -2.0, -1.5], dtype=np.float32)  
loss_fn = nn.BCEWithLogitsLoss()     
manual_way = nn.BCELoss()(torch.sigmoid(torch.tensor(logits)), torch.tensor(y_true_bin, dtype=torch.float32))
combined_way = loss_fn(torch.tensor(logits), torch.tensor(y_true_bin, dtype=torch.float32))
print("Sigmoid then BCE separately:", manual_way.item())
print("BCEWithLogitsLoss (combined):", combined_way.item())

Sigmoid then BCE separately: 0.6517564654350281
BCEWithLogitsLoss (combined): 0.6517564654350281


**Interpretation:** Both approaches give (essentially) the same answer here, but `BCEWithLogitsLoss` is preferred in practice because it's numerically safer — it internally uses a more stable formula that avoids extreme values breaking the `log()` calculation.

**When to use:** always prefer this over manual sigmoid+BCE when building a network in PyTorch — feed it raw logits (no sigmoid in your model's last layer).

**When not to use:** if your model's output layer already includes a sigmoid activation (then use plain `BCELoss`, not this).

**Advantages:** numerically stable, one line instead of two, less chance of bugs.

**Limitations:** requires remembering that the input must be raw logits — accidentally applying sigmoid before this loss will produce incorrect (double-squashed) results.


# Part 3: Multiclass Classification Loss

## 6. Cross Entropy Loss

**Definition:** The multiclass generalization of BCE — measures how well predicted class probabilities (from softmax) match the true class, across **more than 2** classes.

**Formula:** `CE = -log(y_pred[true_class])` (for one example; averaged across all examples)

**Mathematical intuition:** Only the predicted probability assigned to the **correct** class matters — the loss is small if the model gave high probability to the right class, and large if it didn't, regardless of how the remaining probability was split among the wrong classes.

**Simple numerical example:**

In [18]:
true_class = 2                                 
class_probs = np.array([0.1, 0.2, 0.7])     
print("Predicted probability for the TRUE class:", class_probs[true_class])

Predicted probability for the TRUE class: 0.7


**Python implementation:**

In [19]:
def cross_entropy(true_class_probs):
    return -np.log(true_class_probs)
print("Cross entropy loss:", cross_entropy(class_probs[true_class]))

Cross entropy loss: 0.35667494393873245


**PyTorch implementation:** (`nn.CrossEntropyLoss` expects raw logits, and applies softmax internally — like `BCEWithLogitsLoss` did for the binary case)

In [20]:
logits_3class = torch.tensor([[0.5, 1.0, 2.5]])     
true_label = torch.tensor([2])                        
loss_fn = nn.CrossEntropyLoss()
print("Cross Entropy (PyTorch):", loss_fn(logits_3class, true_label).item())

Cross Entropy (PyTorch): 0.30635571479797363


**Interpretation:** Since the model gave a high raw score (2.5) to the true class (index 2), the resulting loss is fairly low. If the true class had instead been index 0 (the lowest score), the loss would be much higher — the loss only cares about the probability mass given to the *correct* answer.

**When to use:** any classification problem with **more than 2** classes (e.g. classifying an order's country among many options), with a `softmax`-style output.

**When not to use:** binary classification (use BCE) or regression (use MAE/MSE/RMSE).

**Advantages:** directly optimizes for putting probability mass on the correct class; works seamlessly with softmax outputs.

**Limitations:** like BCE, can produce very large loss values for confidently wrong predictions, which can occasionally cause unstable training if not managed (e.g. via a lower learning rate).


## Comparing All Losses

In [21]:
summary = {
    'MAE':  ('Regression', 'Average |error|', 'Robust to outliers'),
    'MSE':  ('Regression', 'Average error^2', 'Penalizes large errors, sensitive to outliers'),
    'RMSE': ('Regression', 'sqrt(MSE)', 'Same units as target, still penalizes large errors'),
    'BCE / BCEWithLogits': ('Binary classification', '-log(p) style', 'Punishes confident wrong answers'),
    'Cross Entropy': ('Multiclass classification', '-log(p_true_class)', 'Generalizes BCE to many classes'),
}
for name, (task, formula, note) in summary.items():
    print(f"{name:22} | {task:26} | {formula:20} | {note}")

MAE                    | Regression                 | Average |error|      | Robust to outliers
MSE                    | Regression                 | Average error^2      | Penalizes large errors, sensitive to outliers
RMSE                   | Regression                 | sqrt(MSE)            | Same units as target, still penalizes large errors
BCE / BCEWithLogits    | Binary classification      | -log(p) style        | Punishes confident wrong answers
Cross Entropy          | Multiclass classification  | -log(p_true_class)   | Generalizes BCE to many classes



## Prediction → Loss → Learning

**Explain:** This is the **core training loop** of every neural network:
1. **Prediction:** forward propagation (Notebook 5) turns an input into a predicted output.
2. **Loss:** a loss function compares that prediction to the true answer, producing **one number** describing how wrong it was.
3. **Learning:** that single loss number is used (via backpropagation, next notebook) to compute how each weight should change to make the loss smaller next time — and the optimizer applies that change.

This cycle — predict, measure error, adjust — repeats thousands of times during training, and it's the same 3-step loop no matter which loss function or architecture is used.

**Implement:** trace this chain for one single example.

In [22]:
y_true_example = 1.0
y_pred_example = 0.3                               
loss_value = bce(np.array([y_true_example]), np.array([y_pred_example]))
print(f"Prediction: {y_pred_example}  ->  True label: {y_true_example}  ->  Loss: {loss_value:.4f}")
print("This loss value is exactly what backpropagation (next notebook) will try to reduce.")

Prediction: 0.3  ->  True label: 1.0  ->  Loss: 1.2040
This loss value is exactly what backpropagation (next notebook) will try to reduce.


**Interpret:** We now have the full picture: an input flows forward to become a prediction (Notebook 5), the prediction is compared to the truth to produce a loss (this notebook), and next, that loss will flow **backward** through the network to update every weight — the learning step.